In [1]:
%store -r

In [2]:
import commute_dm.core
import commute_dm.submaps
import commute_dm.utils
import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
def make_backend():
    return momapy_kb.lpg.backends.neo4j.Neo4jBackend(
        hostname=credentials.NEO4J_URI,
        username=credentials.NEO4J_USERNAME,
        password=credentials.NEO4J_PASSWORD,
        notifications_min_severity="off",
    )

In [4]:
MAX_LEVELS = [2, 3, 4, 5, 6]
MIN_N_NODES = 5
UPSTREAM_COLLECTION_NAME = "COVID_DM_CD_AF"
DOWNSTREAM_COLLECTION_NAME = "PD_DM_CD_AF"
INTERFACE_COLLECTION_NAMES = (
    UPSTREAM_COLLECTION_NAME,
    DOWNSTREAM_COLLECTION_NAME,
    "AD_KG_BEL",
)

We compute the interface (between the two activity-flow collections and the AD BEL KG), and load what the sub-maps are made of: the influence structure (two small queries) and every stored map of both activity-flow collections, hydrated as momapy objects. The latter takes about **three minutes** and a couple of hundred megabytes, and is done **once per session** — the sub-maps are then assembled in memory. `4_20`, which builds gene sets and draws nothing, does not load it.

In [ ]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    interface = commute_dm.core.get_interface(session, INTERFACE_COLLECTION_NAMES)
    influences, source_map, node_id_to_object, _, _ = commute_dm.core.load_submap_inputs(
        session, UPSTREAM_COLLECTION_NAME, DOWNSTREAM_COLLECTION_NAME
    )
(
    len(interface),
    len(source_map.model.species),
    len(source_map.model.modulations),
    len(influences.gate_input_node_ids),
)

In [6]:
commute_dm.utils.remake_dir(INTERFACE_ANALYSIS_GRAPHS_DIR)

We assemble and write the sub-maps upstream of the COVID seeds and downstream of the PD seeds. Every element in the output is a **stored** activity-flow element (species, signed modulation, boolean logic gate, glyph, arc) reused as it is, except the synthetic central node standing for the interface protein itself. Species are coloured by which collections they belong to; the third colour marks the species present in **both** disease maps.

In [7]:
with momapy_kb.lpg.session.Session(make_backend()) as session:
    stats_df = commute_dm.core.make_and_write_submaps_from_interface(
        session=session,
        interface=interface,
        influences=influences,
        source_map=source_map,
        node_id_to_object=node_id_to_object,
        output_dir_path=INTERFACE_ANALYSIS_GRAPHS_DIR,
        upstream_collection_name=UPSTREAM_COLLECTION_NAME,
        downstream_collection_name=DOWNSTREAM_COLLECTION_NAME,
        max_levels=MAX_LEVELS,
        min_n_nodes=MIN_N_NODES,
    )
stats_df

,identifier,display_name,max_level,n_species,n_modulations,n_gates,n_compartments,n_templates,n_layout_elements
0,P45983,MAPK8,2,45,56,0,9,47,109
1,P45983,MAPK8,3,94,138,0,13,73,244
2,P45983,MAPK8,4,127,184,0,17,84,327
3,P45983,MAPK8,5,182,285,1,21,134,490
4,P45983,MAPK8,6,252,432,1,28,202,714
...,...,...,...,...,...,...,...,...,...
89,P05231,IL6,2,24,31,0,7,7,61
90,P05231,IL6,3,52,72,1,13,30,140
91,P05231,IL6,4,97,161,1,16,55,277
92,P05231,IL6,5,158,288,2,20,95,472
